# Occluded (partial) colored point-cloud generation

Generates **partial / occluded** colored point clouds from the **dense** ground-truth
clouds produced by `sample_s2` (the 2^20-point colored PLYs), **not** the 8192-point
`sample_s3` downsample. Color (RGB) rides along with geometry the whole way.

**Occlusion logic — ported verbatim from PoinTr** (`utils/misc.py::seprate_point_cloud`,
https://github.com/yuxumin/PoinTr):

1. Normalize the cloud to the unit sphere (PoinTr's data convention).
2. Pick a **viewpoint** = a random unit vector on that sphere
   (`F.normalize(torch.randn(1,1,3))`) — or a fixed one for reproducible test sets.
3. Sort every point by its distance to that viewpoint.
4. **Remove the `num_crop` nearest points** — that contiguous blob is the *missing*
   region to be completed. The remaining points are the *partial* input.

`num_crop = ratio * N`. PoinTr's ShapeNet-55/34 difficulties remove **25% / 50% / 75%**
of the points (simple / moderate / hard). The only change here vs. PoinTr is that each
kept/removed index also carries its RGB, so the partial stays colored.

In [ ]:
import os
# ============================================================
# Cell A - Configuration
# ============================================================
ORTAK = os.environ.get("PCC_DATA_ROOT", os.path.abspath("data"))  # set PCC_DATA_ROOT to your data folder

# --- GROUND TRUTH = the DENSE colored clouds from sample_s2 (2^20 pts), NOT s3's 8192.
# data_s2 holds the raw (speckle) dense clouds. If you have the EPFL de-speckled clean
# dense clouds shared somewhere, point GT_SOURCE_DIR there instead -> higher-quality GT.
GT_SOURCE_DIR = f"{ORTAK}/data_s2"

RUN_TAG = "occ"
OUT_DIR = f"{ORTAK}/occluded_{RUN_TAG}"      # partial (occluded) clouds land here
SAVE_MISSING = True                           # also save the removed region (the "missing" GT)

# PoinTr difficulties = fraction of points REMOVED near a random viewpoint.
RATIOS   = {"simple": 0.25, "moderate": 0.50, "hard": 0.75}
N_VIEWS  = 1              # partials per (model, difficulty); >1 = several random viewpoints
SEED     = 42            # base seed -> deterministic, reproducible partials
PADDING_ZEROS = False    # PoinTr option: True zeros-out the cropped xyz (keeps N fixed)
                         # instead of deleting them. False = drop them (variable-size partial).
TARGET_POINTS = None     # None keeps the dense partial. Set e.g. 16384 to FPS-downsample
                         # the partial (slow on 2^20 pts; see note in Cell C).

WORK_DIR   = os.path.abspath("work_occ")
CATEGORIES = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
SYNSETS    = list(CATEGORIES.values())
NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

In [ ]:
# ============================================================
# Cell B - Mount Drive, install deps, make dirs
# ============================================================
import os, subprocess

subprocess.run("pip install -q -U open3d", shell=True, check=True)

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(GT_SOURCE_DIR), (
    f"\n[STOP] GT source not found:\n  {GT_SOURCE_DIR}\n"
    "Point GT_SOURCE_DIR at your dense sample_s2 output (2^20-pt colored PLYs)."
)
print("Reading dense GT from:", GT_SOURCE_DIR)
print("Writing partials to  :", OUT_DIR)

In [ ]:
# ============================================================
# Cell C1 - Colored-PLY IO helpers (same convention as sample_s3)
# ============================================================
import os
import numpy as np, open3d as o3d

def load_ply_xyzrgb(ply_path):
    """Load a colored PLY -> (N,6) float32 [x,y,z,r,g,b], rgb in [0,1]."""
    pc  = o3d.io.read_point_cloud(ply_path)
    xyz = np.asarray(pc.points)
    rgb = np.asarray(pc.colors)
    if rgb.shape[0] != xyz.shape[0]:          # textureless / colorless -> zeros
        rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], axis=1).astype(np.float32)

def save_ply_xyzrgb(arr, out_ply):
    """Save (N,6) [xyz+rgb] to a colored PLY."""
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(arr[:, :3].astype(np.float64))
    pc.colors = o3d.utility.Vector3dVector(np.clip(arr[:, 3:6], 0, 1).astype(np.float64))
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    o3d.io.write_point_cloud(out_ply, pc)

In [ ]:
# ============================================================
# Cell C2 - PoinTr occlusion, ported to numpy + color
# ------------------------------------------------------------
# Faithful port of PoinTr utils/misc.py::seprate_point_cloud. PoinTr operates on
# xyz only (torch, on GPU, batched); here it is single-cloud numpy and each kept/
# removed index also carries RGB, so the partial stays colored. The geometry is
# identical: viewpoint = random unit vector on the unit sphere, remove the num_crop
# points NEAREST to it, keep the rest.
# ============================================================
import numpy as np

def farthest_point_sampling(points, n_samples, seed=None):
    """Geometric FPS on xyz; color rides along via the returned indices.
    NOTE: pure-python loop -> slow on 2^20 pts. Only used if TARGET_POINTS is set."""
    N = points.shape[0]
    if n_samples >= N:
        return np.arange(N)
    xyz = points[:, :3]
    rng = np.random.default_rng(seed)
    sel = np.zeros(n_samples, dtype=np.int64)
    dist = np.full(N, np.inf)
    sel[0] = rng.integers(N)
    for i in range(1, n_samples):
        d = np.sum((xyz - xyz[sel[i - 1]]) ** 2, axis=1)
        dist = np.minimum(dist, d)
        sel[i] = np.argmax(dist)
    return sel

def separate_point_cloud(points, crop_ratio, viewpoint=None, seed=None, padding_zeros=False):
    """Generate one partial/occluded cloud from a complete colored cloud.

    points        : (N,6) float [xyz + rgb]
    crop_ratio    : fraction of points to REMOVE (PoinTr: 0.25 / 0.50 / 0.75)
    viewpoint     : None -> random unit vector (like PoinTr training);
                    or a 3-vector for a fixed, reproducible viewpoint (test sets)
    padding_zeros : PoinTr option. True -> zero the cropped points' xyz (keep N fixed);
                    False -> delete them (variable-size partial).

    returns (partial, missing), each (M,6). missing is None if padding_zeros=True.
    """
    xyz = points[:, :3]
    N = points.shape[0]
    num_crop = int(round(N * crop_ratio))
    if num_crop <= 0:
        return points.copy(), points[:0].copy()

    # --- normalize to the unit sphere (PoinTr's data convention: viewpoint lives on it)
    c    = xyz.mean(axis=0)
    xyzc = xyz - c
    r    = np.linalg.norm(xyzc, axis=1).max()
    xyzn = xyzc / (r + 1e-12)

    # --- viewpoint = random unit vector  ==  F.normalize(torch.randn(1,1,3))
    rng = np.random.default_rng(seed)
    if viewpoint is None:
        v = rng.standard_normal(3)
        center = v / np.linalg.norm(v)
    else:
        center = np.asarray(viewpoint, dtype=float)
        center = center / np.linalg.norm(center)

    # --- distance to viewpoint, sort ascending; nearest num_crop -> removed
    d   = np.linalg.norm(xyzn - center[None, :], axis=1)
    idx = np.argsort(d)                      # ascending: nearest first
    crop_idx = idx[:num_crop]                # nearest  -> missing region
    keep_idx = idx[num_crop:]                # the rest -> partial input

    if padding_zeros:                        # PoinTr: input_data[crop] *= 0
        partial = points.copy()
        partial[crop_idx, :3] = 0.0
        return partial, None
    return points[keep_idx], points[crop_idx]

In [ ]:
# ============================================================
# Cell D - Smoke test on ONE cloud + visualize the hole (Plotly)
# ------------------------------------------------------------
# Partial shown in its true colors; removed region overlaid semi-transparent red.
# ============================================================
import glob
import numpy as np
import plotly.graph_objects as go

def _sub(a, n):
    if a is None or len(a) <= n:
        return a
    i = np.random.choice(len(a), n, replace=False)
    return a[i]

def show_occlusion(partial, missing, title, max_pts=30000):
    p = _sub(partial, max_pts)
    m = _sub(missing, max_pts)
    pcol = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in p[:, 3:6]]
    traces = [go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode="markers",
                           marker=dict(size=1.5, color=pcol), name="visible (partial)")]
    if m is not None and len(m):
        traces.append(go.Scatter3d(x=m[:, 0], y=m[:, 1], z=m[:, 2], mode="markers",
                      marker=dict(size=1.5, color="red", opacity=0.35), name="removed (missing)"))
    fig = go.Figure(traces)
    fig.update_layout(title=title, height=550, scene_aspectmode="data",
                      margin=dict(l=0, r=0, t=30, b=0))
    fig.show()

_gt_plys = sorted(glob.glob(f"{GT_SOURCE_DIR}/*/*.ply"))
assert _gt_plys, f"No PLYs under {GT_SOURCE_DIR}/*/*.ply"
_src = _gt_plys[0]
_pts = load_ply_xyzrgb(_src)
_partial, _missing = separate_point_cloud(_pts, crop_ratio=0.50, seed=SEED)
print(f"GT: {_pts.shape[0]} pts -> partial: {_partial.shape[0]}  missing: "
      f"{0 if _missing is None else _missing.shape[0]}  (ratio=0.50)")
show_occlusion(_partial, _missing, f"{os.path.basename(_src)} - 50% occluded (PoinTr-style)")

In [ ]:
# ============================================================
# Cell E - Batch: every GT cloud x every difficulty x N_VIEWS
# ------------------------------------------------------------
# Deterministic per-(model, difficulty, view) seed -> reproducible partials.
# Output: OUT_DIR/<difficulty>/<synset>/<model_id>.ply  (+ *_missing.ply).
# ============================================================
import glob, os, traceback
from zlib import crc32
import numpy as np, pandas as pd
from tqdm import tqdm

gt_plys = sorted(glob.glob(f"{GT_SOURCE_DIR}/*/*.ply"))
rows, errors = [], []
diff_names = list(RATIOS)

for src in tqdm(gt_plys, desc="Occluding"):
    synset = os.path.basename(os.path.dirname(src))
    if synset not in SYNSETS:               # skip stray dirs / manifests
        continue
    mid = os.path.splitext(os.path.basename(src))[0]
    try:
        pts = load_ply_xyzrgb(src)
        for di, dname in enumerate(diff_names):
            ratio = RATIOS[dname]
            for v in range(N_VIEWS):
                # stable, reproducible seed from model id + difficulty + view
                seed = SEED + (crc32(mid.encode()) & 0xffff) + di * 104729 + v * 7919
                partial, missing = separate_point_cloud(
                    pts, ratio, seed=seed, padding_zeros=PADDING_ZEROS)

                if TARGET_POINTS and not PADDING_ZEROS and partial.shape[0] > TARGET_POINTS:
                    partial = partial[farthest_point_sampling(partial, TARGET_POINTS, seed=seed)]

                tag = dname if N_VIEWS == 1 else f"{dname}_v{v}"
                out = f"{OUT_DIR}/{tag}/{synset}/{mid}.ply"
                save_ply_xyzrgb(partial, out)

                row = {"synset": synset, "category": NAME_BY_SYNSET[synset],
                       "model_id": mid, "difficulty": tag, "crop_ratio": ratio,
                       "view": v, "seed": seed, "gt_ply": src,
                       "gt_points": int(pts.shape[0]),
                       "partial_ply": out, "partial_points": int(partial.shape[0]),
                       "padding_zeros": PADDING_ZEROS}
                if SAVE_MISSING and missing is not None:
                    mout = f"{OUT_DIR}/{tag}/{synset}/{mid}_missing.ply"
                    save_ply_xyzrgb(missing, mout)
                    row["missing_ply"] = mout
                    row["missing_points"] = int(missing.shape[0])
                rows.append(row)
    except Exception:
        errors.append((synset, mid, traceback.format_exc()))

pd.DataFrame(rows).to_csv(f"{OUT_DIR}/manifest.csv", index=False)
with open(f"{OUT_DIR}/errors.log", "w") as fh:
    fh.write("No errors.\n" if not errors else "")
    for synset, mid, tb in errors:
        fh.write(f"\n=== {synset}/{mid} ===\n{tb}\n")

print(f"Done. {len(rows)} partials from {len(gt_plys)} GT clouds; {len(errors)} failed.")
print(f"Manifest -> {OUT_DIR}/manifest.csv")

In [ ]:
# ============================================================
# Cell F - Sanity check: color survived + counts add up
# ============================================================
import numpy as np
if rows:
    r0 = rows[0]
    p = load_ply_xyzrgb(r0["partial_ply"])
    print("Example:", r0["category"], r0["model_id"], r0["difficulty"])
    print(f"  partial points   : {p.shape[0]}")
    print(f"  has color (std>0): {np.asarray(p[:,3:6]).std() > 0.01}")
    if not r0["padding_zeros"]:
        exp = int(round(r0["gt_points"] * (1 - r0["crop_ratio"])))
        print(f"  kept ~= (1-ratio)*N : {p.shape[0]} vs expected ~{exp}")
    print("Color carried through occlusion: OK" if np.asarray(p[:,3:6]).std() > 0.01
          else "WARNING: partial has no color (was the GT textureless?)")